In [1]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
DATA_PATH = "/content/drive/MyDrive/fraud-risk-platform/Base.csv"
df = pd.read_csv(DATA_PATH)
print(df.shape)

Mounted at /content/drive
(1000000, 32)


In [2]:
sentinel_cols = ['prev_address_months_count', 'bank_months_count',
                  'current_address_months_count', 'session_length_in_minutes',
                  'device_distinct_emails_8w']

# Step 3a: create a "was this unknown" flag for each — this preserves the signal
for col in sentinel_cols:
    df[f'{col}_is_unknown'] = (df[col] == -1).astype(int)

# Step 3b: for tree models (Random Forest/XGBoost), -1 is fine to leave as-is —
# trees naturally learn to split it into its own branch.
# We only need a clean numeric version for Logistic Regression's baseline pipeline,
# so build a separate "imputed" copy for that:
df_for_linear_models = df.copy()
for col in sentinel_cols:
    median_val = df.loc[df[col] != -1, col].median()
    df_for_linear_models[col] = df_for_linear_models[col].replace(-1, median_val)

In [3]:
cat_cols = ['payment_type', 'employment_status', 'housing_status', 'source', 'device_os']
for col in cat_cols:
    print(col, df[col].nunique(), df[col].unique()[:8])

payment_type 5 ['AA' 'AD' 'AB' 'AC' 'AE']
employment_status 7 ['CB' 'CA' 'CC' 'CF' 'CD' 'CE' 'CG']
housing_status 7 ['BC' 'BE' 'BD' 'BA' 'BB' 'BF' 'BG']
source 2 ['INTERNET' 'TELEAPP']
device_os 5 ['linux' 'other' 'windows' 'x11' 'macintosh']


In [4]:
df_encoded = pd.get_dummies(df, columns=cat_cols, drop_first=True)
df_linear_encoded = pd.get_dummies(df_for_linear_models, columns=cat_cols, drop_first=True)
print(df_encoded.shape)

(1000000, 53)


In [5]:
# Velocity acceleration — is spending speeding up compared to the 24h average?
df_encoded['velocity_accel_6h_vs_24h'] = df_encoded['velocity_6h'] - (df_encoded['velocity_24h'] / 4)

# Income relative to requested credit — asking for a lot relative to income is a risk signal
df_encoded['credit_to_income_ratio'] = df_encoded['proposed_credit_limit'] / (df_encoded['income'] + 0.01)

# Apply the same two to the linear-model dataframe for consistency
df_linear_encoded['velocity_accel_6h_vs_24h'] = df_linear_encoded['velocity_6h'] - (df_linear_encoded['velocity_24h'] / 4)
df_linear_encoded['credit_to_income_ratio'] = df_linear_encoded['proposed_credit_limit'] / (df_linear_encoded['income'] + 0.01)

In [6]:
train_df = df_encoded[df_encoded['month'] <= 5]
val_df   = df_encoded[df_encoded['month'] == 6]
test_df  = df_encoded[df_encoded['month'] == 7]

print("Train:", train_df.shape, "| Val:", val_df.shape, "| Test:", test_df.shape)
print("Train fraud rate:", train_df['fraud_bool'].mean()*100, "%")
print("Val fraud rate:  ", val_df['fraud_bool'].mean()*100, "%")
print("Test fraud rate: ", test_df['fraud_bool'].mean()*100, "%")

Train: (794989, 55) | Val: (108168, 55) | Test: (96843, 55)
Train fraud rate: 1.0252972053701372 %
Val fraud rate:   1.3405073589231566 %
Test fraud rate:  1.4745515938167963 %


In [7]:
import os
os.makedirs('/content/drive/MyDrive/fraud-risk-platform/processed', exist_ok=True)

train_df.to_csv('/content/drive/MyDrive/fraud-risk-platform/processed/train.csv', index=False)
val_df.to_csv('/content/drive/MyDrive/fraud-risk-platform/processed/val.csv', index=False)
test_df.to_csv('/content/drive/MyDrive/fraud-risk-platform/processed/test.csv', index=False)